# E5 — Normal-aware loss full-stage (UNet 224×224)

Attach the preprocessed BTXRD Dataset, the E4 screening archive/output, and—after the first run—the previous E5 full-stage archive/output. Internet must be enabled to clone the branch. Each saved version runs exactly one (loss, seed) job, imports previous checkpoints, then emits a new full-stage archive.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import torch
import yaml

REPO_URL = 'https://github.com/lehngoc/BTXRD-LViT.git'
BRANCH = 'model/loss-ablation-normal-fp'
REPO_ROOT = Path('/kaggle/working/BTXRD-LViT')
WORK_ROOT = Path('/kaggle/working/experiments/loss_ablation')

assert torch.cuda.is_available(), 'Enable a T4 GPU in Kaggle Notebook Settings.'
if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)
subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'albumentations', 'PyYAML'], check=True)

In [ ]:
def find_data_root() -> Path:
    suffix = 'data/exports/btxrd_preprocessed/train.csv'
    for csv_path in Path('/kaggle/input').rglob('train.csv'):
        if csv_path.as_posix().endswith(suffix):
            return csv_path.parents[3]
    raise FileNotFoundError('Missing attached BTXRD preprocessed Dataset.')

def find_screening_summary() -> Path:
    matches = list(Path('/kaggle/input').rglob('screening_loss_ablation_summary.json'))
    if matches:
        return matches[0]

    # Fallback: E4 artifacts uploaded as a Kaggle Dataset archive.
    archives = list(Path('/kaggle/input').rglob('loss_ablation_screening_artifacts.tar.gz'))
    archives += list(Path('/kaggle/input').rglob('loss_ablation_screening_artifacts.zip'))
    if not archives:
        raise FileNotFoundError('Attach E4 notebook output or the E4 screening artifacts archive Dataset.')
    extract_root = Path('/kaggle/working/imported_screening')
    if extract_root.exists():
        shutil.rmtree(extract_root)
    shutil.unpack_archive(str(archives[0]), str(extract_root))
    matches = list(extract_root.rglob('screening_loss_ablation_summary.json'))
    if not matches:
        raise FileNotFoundError(f'Archive does not contain screening summary: {archives[0]}')
    return matches[0]

DATA_ROOT = find_data_root()
screening_summary_path = find_screening_summary()
screening_root = screening_summary_path.parent
WORK_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copytree(screening_root, WORK_ROOT / 'screening', dirs_exist_ok=True)
screening = json.loads(screening_summary_path.read_text(encoding='utf-8'))
# E4 showed bce_dice_07 dominates Focal on every primary metric, so use the
# compute-saving protocol: baseline versus the single screening winner.
selected_challengers = ['bce_dice_07']
full_losses = ['bce_dice_05'] + selected_challengers
print('Full-stage losses:', full_losses)

def import_previous_full_artifacts() -> bool:
    matches = list(Path('/kaggle/input').rglob('full_loss_ablation_summary.json'))
    if matches:
        source_root = matches[0].parent
    else:
        archives = list(Path('/kaggle/input').rglob('loss_ablation_full_artifacts.tar.gz'))
        archives += list(Path('/kaggle/input').rglob('loss_ablation_full_artifacts.zip'))
        if not archives:
            return False
        extract_root = Path('/kaggle/working/imported_full')
        if extract_root.exists():
            shutil.rmtree(extract_root)
        shutil.unpack_archive(str(archives[0]), str(extract_root))
        matches = list(extract_root.rglob('full_loss_ablation_summary.json'))
        if not matches:
            raise FileNotFoundError(f'Archive does not contain full-stage summary: {archives[0]}')
        source_root = matches[0].parent

    shutil.copytree(source_root, WORK_ROOT / 'full', dirs_exist_ok=True)
    return True

print('Imported previous E5 artifacts:', import_previous_full_artifacts())

In [ ]:
RUNTIME_DIR = Path('/kaggle/working/runtime_configs/full')
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
source_by_loss = {}
for source in (REPO_ROOT / 'configs/loss_ablation').glob('*.yaml'):
    cfg = yaml.safe_load(source.read_text(encoding='utf-8'))
    source_by_loss[cfg['experiment']['loss_id']] = source

def make_runtime_config(loss_id: str, seed: int) -> Path:
    cfg = yaml.safe_load(source_by_loss[loss_id].read_text(encoding='utf-8'))
    cfg['data']['root_dir'] = str(DATA_ROOT)
    cfg['training']['device'] = 'cuda'
    cfg['training']['num_workers'] = 2
    cfg['training']['epochs'] = 200
    cfg['training']['seed'] = seed
    cfg['training']['output_dir'] = str(WORK_ROOT / 'full' / loss_id / f'seed{seed}')
    destination = RUNTIME_DIR / loss_id / f'seed{seed}.yaml'
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    return destination

# Change only this line for the next Kaggle session. Keep the prescribed order:
# all baseline seeds first, then the matching challenger seed.
RUN_TO_EXECUTE = ('bce_dice_05', 42)
loss_id, seed = RUN_TO_EXECUTE
assert loss_id in full_losses, f'Unknown/unselected loss: {loss_id}'
assert seed in [42, 52, 62, 72, 82], seed

run_dir = WORK_ROOT / 'full' / loss_id / f'seed{seed}'
log_dir = Path('/kaggle/working/run_logs')
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / f'{loss_id}_seed{seed}.log'
if (run_dir / 'best_summary.json').exists():
    print(f'Skipping completed run: {loss_id}, seed={seed}')
else:
    if loss_id != 'bce_dice_05':
        baseline_summary = WORK_ROOT / 'full' / 'bce_dice_05' / f'seed{seed}' / 'best_summary.json'
        assert baseline_summary.exists(), f'Run baseline seed {seed} before challenger seed {seed}.'
    runtime_config = make_runtime_config(loss_id, seed)
    print(f'===== full: {loss_id}, seed={seed} =====')
    print(f'Training log is written to: {log_path}')
    with log_path.open('w', encoding='utf-8') as log_file:
        completed = subprocess.run(
            [sys.executable, '-m', 'src.training.train_unet', '--config', str(runtime_config)],
            cwd=REPO_ROOT, stdout=log_file, stderr=subprocess.STDOUT, text=True,
        )
    tail = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-40:]
    print('\n'.join(tail))
    if completed.returncode != 0:
        raise RuntimeError(f'Training failed; inspect {log_path}')

In [ ]:
full_root = WORK_ROOT / 'full'
subprocess.run([
    sys.executable, '-m', 'src.training.aggregate_loss_ablation',
    '--runs-root', str(full_root), '--stage', 'full',
], cwd=REPO_ROOT, check=True)

summary = json.loads((full_root / 'full_loss_ablation_summary.json').read_text(encoding='utf-8'))
print(json.dumps({
    'best_challenger': summary['best_challenger'],
    'validation_winner_overall': summary['validation_winner_overall'],
}, indent=2))

# Archive only completed full-stage runs; E4 screening artifacts are already
# supplied separately as an Input and would otherwise be copied every session.
archive = shutil.make_archive('/kaggle/working/loss_ablation_full_artifacts', 'gztar',
                              root_dir='/kaggle/working/experiments/loss_ablation', base_dir='full')
print('Saved archive:', archive)